# F1 Pit Stop Prediction — Annotated ML Notebook

This notebook walks through a complete machine learning pipeline to predict **when an F1 driver will pit**, using lap-by-lap timing data from the FastF1 library.

**Problem framing:** For every lap of every race, we ask: *did a pit stop happen on this lap?*  
This is a **binary classification** problem — the target is 1 (pit) or 0 (no pit).

**What you'll need:**
```
pip install fastf1 pandas scikit-learn matplotlib seaborn
```

---
## Step 1 — Load race data

FastF1 connects to the official F1 timing feed and downloads lap-by-lap data for every driver in a session.

We loop over multiple seasons so the model sees many different circuits, weather conditions, and team strategies. More races = more diverse training data = a more general model.

> **Caching:** The first call for each session downloads ~10MB of data. After that, it's instant from the local cache. Always enable the cache.

In [ ]:
import fastf1
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Enable local caching so we don't re-download data on every run
fastf1.Cache.enable_cache('f1_cache')

all_laps = []

for year in [2021, 2022, 2023]:
    schedule = fastf1.get_event_schedule(year)
    
    for _, event in schedule.iterrows():
        try:
            # Load the Race session (not qualifying or practice)
            session = fastf1.get_session(year, event['EventName'], 'R')
            session.load(telemetry=False, weather=False)  # Skip telemetry for speed
            
            laps = session.laps.copy()
            laps['Year'] = year
            laps['EventName'] = event['EventName']
            all_laps.append(laps)
            
        except Exception as e:
            # Some sessions (e.g. cancelled races) may fail — skip them
            print(f"Skipping {year} {event['EventName']}: {e}")
            continue

# Combine all seasons into one DataFrame
df = pd.concat(all_laps, ignore_index=True)

print(f"Total laps loaded: {len(df):,}")
print(f"Columns: {list(df.columns)}")

---
## Step 2 — Create the target variable

The target label answers: *did this lap end with a pit stop?*

FastF1 marks pit-out laps with a `PitOutTime` timestamp. If that field is not null, a pit stop happened. We convert this to a simple 1/0 integer.

> **Class imbalance warning:** In a 70-lap race, typically only 2–3 laps are pit laps. So roughly **96% of rows will be 0**. A naive model that always predicts "no pit" would be 96% accurate but completely useless. We handle this in Step 5 with `class_weight='balanced'`.

In [ ]:
# If PitOutTime is not null, a pit stop occurred on that lap
df['pit_this_lap'] = df['PitOutTime'].notna().astype(int)

# Check the imbalance
pit_counts = df['pit_this_lap'].value_counts()
print("Pit stop distribution:")
print(pit_counts)
print(f"\nPit stop rate: {pit_counts[1] / len(df) * 100:.1f}%")

---
## Step 3 — Engineer features

Raw columns like `LapTime` and `TyreLife` are useful, but we can derive better signals from them.

**Feature engineering** means creating new columns that more directly capture the strategic situation on each lap.

| Feature | What it measures | Why it matters |
|---|---|---|
| `tire_age` | Laps on current set of tires | Older tires = more likely to pit |
| `lap_delta` | Lap time vs driver's rolling 3-lap average | Rising delta = tire degradation |
| `race_lap_pct` | How far through the race (0–1) | Captures strategic pit windows |
| `compound_enc` | Soft/Medium/Hard as 0/1/2 | Different compounds last different lengths |

> **Note on `lap_delta`:** We use a rolling average rather than comparing to a fixed baseline, because track conditions and fuel load change across the race. The rolling average adapts to those shifts and isolates the degradation signal.

In [ ]:
# --- Feature 1: Tire age ---
# TyreLife is the number of laps already driven on this compound
df['tire_age'] = df['TyreLife'].fillna(0)

# --- Feature 2: Lap time delta (degradation signal) ---
# Convert LapTime from timedelta to seconds
df['lap_time_sec'] = df['LapTime'].dt.total_seconds()

# Rolling 3-lap average per driver per race — captures their "normal" pace
df['rolling_avg'] = (
    df.groupby(['Year', 'EventName', 'DriverNumber'])['lap_time_sec']
      .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

# Positive delta = slower than average = degrading tires
df['lap_delta'] = df['lap_time_sec'] - df['rolling_avg']

# --- Feature 3: Race progress (0 = start, 1 = finish) ---
# Pit windows typically open 20–30% into a race
df['race_lap_pct'] = (
    df['LapNumber'] /
    df.groupby(['Year', 'EventName'])['LapNumber'].transform('max')
)

# --- Feature 4: Tire compound encoded as integer ---
compound_map = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2, 'INTERMEDIATE': 3, 'WET': 4}
df['compound_enc'] = df['Compound'].map(compound_map)

# Define which columns are features and which is the target
FEATURES = ['tire_age', 'lap_delta', 'race_lap_pct', 'compound_enc']
TARGET   = 'pit_this_lap'

# Drop rows with missing values in any feature column
df_clean = df.dropna(subset=FEATURES + [TARGET]).copy()

print(f"Clean dataset: {len(df_clean):,} laps")
df_clean[FEATURES + [TARGET]].head(10)

---
## Step 4 — Split into train and test sets

We hold out the **entire 2023 season** as the test set and train on 2021–2022.

**Why split by season rather than randomly?**  
Laps from the same race are highly correlated — lap 30 and lap 31 of the same race share similar tire age, fuel load, and track conditions. A random split would put both in train and test, making evaluation results look better than they actually are. This is called **data leakage**.

Splitting by season gives a clean temporal boundary: the model has truly never seen any 2023 race data.

In [ ]:
# Train on 2021–2022, test on 2023 (unseen season)
train = df_clean[df_clean['Year'] < 2023]
test  = df_clean[df_clean['Year'] == 2023]

X_train = train[FEATURES]
y_train = train[TARGET]

X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"Training laps : {len(X_train):,}")
print(f"Test laps     : {len(X_test):,}")
print(f"\nPit rate in train: {y_train.mean()*100:.1f}%")
print(f"Pit rate in test : {y_test.mean()*100:.1f}%")

---
## Step 5 — Train a Random Forest classifier

A **Random Forest** builds many decision trees, each trained on a random subset of the data and features. To make a prediction, all trees vote and the majority wins.

It's the right first model here because:
- It handles tabular data well without feature scaling
- It's robust to outliers and noise
- It provides built-in **feature importance** scores
- It rarely overfits badly with default settings

Key parameter: `class_weight='balanced'` tells the model to weight pit-stop laps more heavily during training, compensating for the fact that they're rare (only ~4% of laps). Without this, the model tends to ignore the minority class entirely.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,        # Number of trees — more = more stable, but slower
    max_depth=8,             # Limit tree depth to prevent overfitting
    class_weight='balanced', # Compensate for pit-stop class imbalance
    random_state=42,         # For reproducibility
    n_jobs=-1                # Use all CPU cores
)

model.fit(X_train, y_train)

print("Model trained successfully.")
print(f"Number of trees: {model.n_estimators}")
print(f"Features used  : {FEATURES}")

---
## Step 6 — Evaluate on the 2023 test season

**Don't use accuracy as your primary metric for imbalanced data.**  
A model that always predicts "no pit" would be ~96% accurate but catch zero pit stops.

Instead, focus on:
- **Precision** — of all laps the model flagged as pits, what fraction actually were? (Avoids false alarms)
- **Recall** — of all actual pit laps, what fraction did the model catch? (Avoids missed stops)
- **F1 score** — harmonic mean of precision and recall; useful single-number summary

In a real strategy tool, you'd tune the classification **threshold** (default 0.5) to balance these: a lower threshold catches more pit stops but raises more false alarms.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of pit = 1

print("Classification report (2023 test season):")
print(classification_report(y_test, y_pred, target_names=['No pit', 'Pit']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: No pit', 'Pred: Pit'],
            yticklabels=['Actual: No pit', 'Actual: Pit'], ax=ax)
ax.set_title('Confusion matrix — 2023 test season')
plt.tight_layout()
plt.show()

---
## Step 7 — Feature importance

One of the best things about Random Forest is that it tells you **which features contributed most to its decisions**.

This is called **feature importance** and is measured by how much each feature reduces uncertainty (Gini impurity) across all trees.

Expected result:
- `tire_age` should dominate — it's the clearest signal for when a pit is due
- `lap_delta` should be second — it captures the lap time slowdown from degradation
- `race_lap_pct` captures strategic windows (pits rarely happen on lap 1 or the final lap)
- `compound_enc` has some signal — softs are pitted earlier than hards

> This is satisfying because it matches what real F1 strategists track. When your ML model agrees with domain expertise, it's a good sign it's learned something real.

In [ ]:
import numpy as np

importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_features = [FEATURES[i] for i in indices]
sorted_importances = importances[indices]

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#7F77DD', '#1D9E75', '#EF9F27', '#888780']
bars = ax.barh(sorted_features[::-1], sorted_importances[::-1],
               color=colors[::-1], edgecolor='none')

# Add percentage labels
for bar, val in zip(bars, sorted_importances[::-1]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val*100:.1f}%', va='center', fontsize=11)

ax.set_xlabel('Feature importance')
ax.set_title('What drove the model\'s pit stop predictions?')
ax.set_xlim(0, max(importances) * 1.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Step 8 — Visualise predictions on a single race

Aggregate metrics are useful, but it's also worth looking at how the model performs on a specific race — does its confidence spike at the right laps?

Here we plot pit stop probability across all laps of one race for one driver. The orange spikes should roughly align with the actual pit stop laps (vertical dashed lines).

In [ ]:
# Pick a specific race and driver to inspect
RACE    = 'Bahrain Grand Prix'
YEAR    = 2023
DRIVER  = 'VER'  # Max Verstappen

race_df = df_clean[
    (df_clean['Year'] == YEAR) &
    (df_clean['EventName'] == RACE) &
    (df_clean['Abbreviation'] == DRIVER)
].sort_values('LapNumber')

if len(race_df) == 0:
    print(f"No data found for {DRIVER} at {RACE} {YEAR}. Try a different driver abbreviation.")
else:
    pit_probs = model.predict_proba(race_df[FEATURES])[:, 1]
    actual_pit_laps = race_df[race_df['pit_this_lap'] == 1]['LapNumber'].values

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.fill_between(race_df['LapNumber'], pit_probs, alpha=0.3, color='#EF9F27')
    ax.plot(race_df['LapNumber'], pit_probs, color='#EF9F27', lw=1.5,
            label='Pit probability')

    for lap in actual_pit_laps:
        ax.axvline(lap, color='#E24B4A', linestyle='--', lw=1.5,
                   label='Actual pit' if lap == actual_pit_laps[0] else '')

    ax.axhline(0.5, color='gray', linestyle=':', lw=1, label='Decision threshold (0.5)')
    ax.set_xlabel('Lap number')
    ax.set_ylabel('Pit stop probability')
    ax.set_title(f'Model confidence — {DRIVER}, {RACE} {YEAR}')
    ax.set_ylim(0, 1)
    ax.legend()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

---
## Next steps

You now have a working pit stop prediction model. Here are some directions to extend it:

**Better features:**
- `gap_to_leader` — drivers under pressure may pit earlier to avoid being undercut
- `safety_car_laps` — safety cars trigger opportunistic pit stops
- `track_temp` — hot tracks degrade tires faster
- `laps_behind_leader` — whether a driver is lapping or being lapped

**Better models:**
- `XGBoostClassifier` — often outperforms Random Forest on tabular data
- `LogisticRegression` — simpler, more interpretable
- Tune the decision threshold (try 0.3–0.4 instead of 0.5) to improve recall

**Better evaluation:**
- Cross-validate across multiple held-out seasons
- Evaluate separately by circuit type (street circuits vs permanent tracks)
- Plot a precision-recall curve to find the optimal threshold

**Going further:**
- Extend to predict tire compound choice (multi-class classification)
- Build a race simulator that uses model predictions to compare strategy outcomes
- Add real-time data from OpenF1 to score live laps during a race